# Binary Classification and Feature Importance 
- 8170 positive and negative samples 
- 142501 unseen samples 

In [40]:
import pandas as pd
from tqdm import tqdm 
import os 
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from sklearn.feature_selection import SelectFromModel
from pprint import pprint
import joblib

In [124]:
ground = pd.read_csv("../feature_vectors/ground.csv")
all_sermons = pd.read_csv("../feature_vectors/sermons_1.csv")
all_unknown = pd.read_csv("../feature_vectors/unknown_1.csv")
for file in tqdm(os.listdir("../feature_vectors")): 
    if "sermons" in file and "_1.csv" not in file: 
        all_sermons = pd.concat([all_sermons, pd.read_csv(f"../feature_vectors/{file}")])
    if "unknown" in file and "_1.csv" not in file: 
        all_unknown = pd.concat([all_unknown, pd.read_csv(f"../feature_vectors/{file}")])
print(len(all_sermons), len(all_unknown))
everything = pd.concat([ground,all_sermons]).fillna(0)
everything = pd.concat([everything,all_unknown]).fillna(0)
everything

  0%|          | 0/87 [00:00<?, ?it/s]/var/folders/3_/ylrg8wdj20l755p4921q0wg80000gp/T/ipykernel_52579/481142393.py:8: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  all_unknown = pd.concat([all_unknown, pd.read_csv(f"../feature_vectors/{file}")])
  7%|▋         | 6/87 [00:01<00:27,  2.95it/s]/var/folders/3_/ylrg8wdj20l755p4921q0wg80000gp/T/ipykernel_52579/481142393.py:8: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  all_unknown = pd.concat([all_unknown, pd.read_csv(f"../feature_vectors/{file}")])
100%|██████████| 87/87 [01:33<00:00,  1.08s/it]


19010 148718


,tcpID,curr_div_idx,type_token_ratio,COUNT_tokens,sentence_length,content,function,punctuation,female_pronouns,male_pronouns,...,#,“,****************************,$,․,#&,%,¡,•§,##
0,DDC_FINAL,1,0.336142,2374,0.009088,0.503370,0.395114,0.101516,0.000000,0.333333,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,DDC_FINAL,3,0.292659,2057,0.005678,0.489062,0.331551,0.179387,0.000000,0.560000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,DDC_FINAL,5,0.179240,6628,0.001557,0.501811,0.293754,0.204436,0.000000,0.810345,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,DDC_FINAL,6,0.198991,4558,0.005316,0.519526,0.350154,0.130320,0.000000,0.541176,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,DDC_FINAL,7,0.179376,5285,0.002837,0.505393,0.319395,0.175213,0.000000,0.754098,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1103,A85713,5 | 23226,0.338162,2209,0.020214,0.477592,0.338162,0.184246,0.000000,0.444444,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1104,A85713,6,0.250499,5513,0.008945,0.482133,0.344640,0.173227,0.010000,0.480000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1105,A85713,6 | 23247,0.260632,6043,0.007453,0.488830,0.344862,0.166308,0.042254,0.471831,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1106,A85713,6 | 23271,0.344113,2752,0.014216,0.506177,0.338663,0.155160,0.086420,0.555556,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [125]:
UNLABELED = pd.read_csv("../CORPUS/APS_METADATA_UNLABELED.csv").fillna('')  
UNLABELED 

,id,title,authors,publisher,pubplace,date,subject_heading,group
0,B15663,The death of the tvvo renowned kings of Sweden...,"T. H., fl. 1633.; Schloer, Frederike.",Printed by I D awson for Nicholas Bourne and a...,London,1633,"Sermons, English -- 17th century.",sermon
1,B10040,The perfection of justification maintained aga...,"Simpson, John, 17th cent.",Printed by M Simmons for Hanna Allen and are t...,London,1648,"Sermons, English -- Early works to 1800.; San...",sermon
2,B09423,Speech delivered at a visitation held in the D...,"Loftus, Dudley, 1619-1695.",Printed by Benjamin Tooke Printer to the King ...,Dublin,1671,Presbyterian Church -- Controversial literatu...,sermon
3,B07676,"The earths encrease. Or, a communion cup: pres...","Field, Theophilus, 1574-1636.",by Miles Flesher for Nath Feild sic,Printed at London,1624,"Bible. -- O.T. -- Psalms LXVII, 6 -- Sermon...",sermon
4,A87420,Enochs walk and change opened in a sermon at L...,"Jacombe, Thomas, 1622-1687.",Printed by T R and E M for Ralph Smith,London,1656,"Vines, Richard, 1600?-1656.; Funeral sermons.;...",sermon
...,...,...,...,...,...,...,...,...
8200,A20184,"The hand-maid of repentance. Or, A short treat...","Dent, Arthur, d. 1607.; Dent, Arthur, d. 1607....",Printed by G Eld for Thomas Thorp,London,1614,Restitution -- Early works to 1800.,unknown
8201,A25836,"The Army brought to the barre, legally examine...",Andrew All Truth.,s n,London,1647,England and Wales. -- Army.; England and Wale...,unknown
8202,A20088,"A tragi-comedy: called, Match mee in London As...","Dekker, Thomas, ca. 1572-1632.",Printed by B Alsop and T Favvcet for H Seile a...,London,1631,,unknown
8203,A25723,The history of Appian of Alexandria in two par...,"Davies, John, 1625-1693.; Appianus, of Alexand...",Printed for John Amery,London,1679,Rome -- History.,unknown


In [126]:
LABELED = pd.read_csv("../CORPUS/APS_METADATA_ground.csv").fillna('')
LABELED['group'] = "positive"
unlabeled_tcpIDs = set(UNLABELED['id'].to_list())
unknown = pd.read_csv("../CORPUS/APS_METADATA_unknown.csv").fillna('')  
unknown = unknown[~unknown['id'].isin(unlabeled_tcpIDs)]
unknown['group'] = 'negative'
LABELED = pd.concat([LABELED, unknown])
LABELED

,id,title,authors,publisher,pubplace,date,subject_heading,group
0,DDC_FINAL,"A Treatise of Christian Doctrine, compiled fro...","Milton, John, 1608-1674.; Sumner, Charles Rich...",Cambridge University Press,Cambridge,1825,"Theology, Doctrinal.",positive
1,B08803,Several discourses concerning the actual Provi...,"Collinges, John, 1623-1690.",Printed for Tho Parkhurst and are to be sold b...,London,1678,Providence and government of God -- Sermons -...,positive
2,B08178,The treasure of true loue or A liuely descript...,"Tuke, Thomas, d. 1657.",Printed by Thomas Creede and are to be solde b...,London,1608,Jesus Christ -- Person and offices -- Early ...,positive
3,A95924,Theoremata theologica: = Theological treatises...,"Vilvain, Robert, 1575?-1663.",Printed by R Hodgkinsonne for the author and a...,London,1654,"Theology, Doctrinal -- Early works to 1800.",positive
4,A91437,The late Assembly of Divines Confession of fai...,"Parker, William, fl. 1651-1658.",s n,London,1651,Church of England -- Doctrines -- Early work...,positive
...,...,...,...,...,...,...,...,...
8375,A29139,"A true relation of the proceedings, examinatio...","Buckley, Francis, Gent.",Printed for Daniel Pakeman,London,1660,"Bradshaw, John, 1602-1659.; Andrews, Eusebius,...",negative
8381,A29623,Songs and other poems by Alex. Brome ...,"Brome, Alexander, 1620-1666.",Printed for Henry Brome,London,1664,,negative
8426,A20435,The coppie of the Anti-Spaniard made at Paris ...,"Arnauld, Antoine, 1560-1619, attributed name.;...",printed by Iohn Wolfe,London,1590,"Philip -- II, -- King of Spain, 1527-1598 --...",negative
8443,A20051,"The blacke rod, and the vvhite rod (justice an...","Dekker, Thomas, ca. 1572-1632.",Printed by B A and T F for Iohn Covvper,London,1630,Plague -- England -- Early works to 1800.,negative


In [127]:
def convert_df(df): 
    return df.drop(["tcpID",'curr_div_idx','COUNT_tokens'],axis=1)

# metadata 
pos_df = LABELED[LABELED['group'] == "positive"]
neg_df = LABELED[LABELED['group'] == "negative"]
positive = set(pos_df['id'].to_list())
negative = set(neg_df['id'].to_list())
unlabeled = set(UNLABELED['id'].to_list())

# embeddings 
pos_vec_df = everything[everything['tcpID'].isin(positive)]
neg_vec_df = everything[everything['tcpID'].isin(negative)]
pos_vectors = convert_df(pos_vec_df).to_numpy()
neg_vectors = convert_df(neg_vec_df).to_numpy()
unlabeled_df = everything[everything['tcpID'].isin(unlabeled)]
unlabeled_vectors = convert_df(unlabeled_df).to_numpy()
print(len(pos_vectors),len(neg_vectors),len(unlabeled_vectors))

3327 5755 161973


In [129]:
labeled_dict = pd.concat([pos_vec_df, neg_vec_df]).to_dict(orient='records')
labeled_dict = {i: item for i, item in enumerate(labeled_dict)}
unlabeled_dict = {i: item for i, item in enumerate(unlabeled_df.to_dict(orient='records'))}
labeled_metadata_dict = {item['id']:item for item in LABELED.to_dict(orient='records')}
unlabeled_metadata_dict = {item['id']:item for item in UNLABELED.to_dict(orient='records')}
len(labeled_dict), len(unlabeled_dict),len(labeled_metadata_dict), len(unlabeled_metadata_dict)

(9082, 161973, 384, 8205)

## Classify & analyze

In [131]:
X = np.vstack([pos_vectors, neg_vectors])
y = np.hstack([np.ones(len(pos_vectors)), np.zeros(len(neg_vectors))])

# 80-20 train-test split 
idx = np.arange(len(X))
X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(X, y, idx, test_size=0.20, stratify=y, random_state=42)

classifier = RandomForestClassifier(
    n_estimators=500, # why 500 estimators? 
    max_depth=None,
    n_jobs=-1,
    random_state=42,
    # class_weight="balanced" 
)

In [132]:
test_df = []
for i in idx_test: 
    tcpID = labeled_dict[i]['tcpID']
    curr_div_idx = labeled_dict[i]['curr_div_idx']
    test_df.append({'tcpID':tcpID, 'curr_div_idx':curr_div_idx})
test_df = pd.DataFrame(test_df)
test_df.to_csv("../OUTPUTS/test.csv",index=False)
test_df

,tcpID,curr_div_idx
0,A65910,19 | 32488
1,A15447,40
2,A53061,113
3,DDC_FINAL,19
4,B11899,5 | 5154
...,...,...
1812,A34471,15
1813,A15447,36 | 35842
1814,A09339,81
1815,A26864,60


In [169]:
9082-1817

7265

In [133]:
# cross validation 
cross_valid = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cross_valid_scores = cross_val_score(classifier, X_train, y_train, cv=cross_valid, scoring="roc_auc", n_jobs=-1)
print("Cross validation scores:", cross_valid_scores)
print("Cross validation mean:", cross_valid_scores.mean())

Cross validation scores: [0.99800703 0.99598855 0.99578343 0.99676206 0.99631903]
Cross validation mean: 0.9965720197137815


In [134]:
classifier.fit(X_train, y_train)

y_pred = classifier.predict(X_test)
y_proba = classifier.predict_proba(X_test)[:, 1]
print(classification_report(y_test, y_pred))
print("ROC AUC (test):", roc_auc_score(y_test, y_proba))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))

joblib.dump(classifier, "../OUTPUTS/rf_model.joblib")

              precision    recall  f1-score   support

         0.0       0.98      0.99      0.98      1151
         1.0       0.97      0.97      0.97       666

    accuracy                           0.98      1817
   macro avg       0.98      0.98      0.98      1817
weighted avg       0.98      0.98      0.98      1817

ROC AUC (test): 0.9974855394056089
Confusion matrix:
 [[1134   17]
 [  20  646]]


['../OUTPUTS/rf_model.joblib']

In [135]:
features = convert_df(everything[everything['tcpID'].isin(positive)]).columns
features = list(features)
len(features)

1035

In [168]:
misclassified_df = []
for idx, pred in enumerate(y_pred): 
    orig_idx = idx_test[idx]
    tcpID = labeled_dict[orig_idx]['tcpID']
    curr_div_idx = labeled_dict[orig_idx]['curr_div_idx']
    correct = y_test[idx]
    if correct != pred: 
        misclassified_df.append({'tcpID':tcpID, 'curr_div_idx':curr_div_idx, 'correct_label':correct, 'prediction':pred})
misclassified_df = pd.DataFrame(misclassified_df)
misclassified_df.to_csv("../OUTPUTS/misclassified_test.csv",index=False)
misclassified_df

,tcpID,curr_div_idx,correct_label,prediction
0,A75849,12 | 24471,1.0,0.0
1,A29133,3,1.0,0.0
2,A85770,15,0.0,1.0
3,A91437,41,1.0,0.0
4,A56589,12 | 32340,1.0,0.0
5,A52417,89,0.0,1.0
6,A18974,33,0.0,1.0
7,A69728,1,1.0,0.0
8,A51310,54,0.0,1.0
9,A56589,11 | 32110,1.0,0.0


In [ ]:
# Inference on unseen data
preds_unseen = classifier.predict(unlabeled_vectors)

In [151]:
preds_unseen

array([1., 1., 1., ..., 0., 0., 0.])

In [155]:
pos_pred_df = []
neg_pred_df = []
for i, pred in enumerate(preds_unseen): 
    tcpID = unlabeled_dict[i]['tcpID']
    curr_div_idx = unlabeled_dict[i]['curr_div_idx']
    group = unlabeled_metadata_dict[tcpID]['group']
    if int(pred) == 1: 
        pos_pred_df.append({'tcpID':tcpID, 'curr_div_idx':curr_div_idx,'group':group})
    else: 
        neg_pred_df.append({'tcpID':tcpID, 'curr_div_idx':curr_div_idx,'group':group})
pos_pred_df = pd.DataFrame(pos_pred_df)
pos_pred_df.to_csv("../OUTPUTS/PREDICTIONS_POSITIVE.csv",index=False)
neg_pred_df = pd.DataFrame(neg_pred_df)
neg_pred_df.to_csv("../OUTPUTS/PREDICTIONS_NEGATIVE.csv",index=False)

print(len(pos_pred_df[pos_pred_df['group']=="sermon"])," positive sermon sections")
sermons = UNLABELED[UNLABELED['group']=="sermon"]['id']
print(len(unlabeled_df[unlabeled_df['tcpID'].isin(sermons)])," total sermon sections")
print(len(pos_pred_df[pos_pred_df['group']=="unknown"])," positive other sections")
print(len(unlabeled_df[~unlabeled_df['tcpID'].isin(sermons)])," total other sections")
print(len(pos_pred_df)," total positive sections")
print(len(neg_pred_df)," total negative sections")
print(len(set(pos_pred_df['tcpID'])),' total tcpIDs containing positive predictions ')
pos_pred_df

13785  positive sermon sections
15480  total sermon sections
67513  positive other sections
146493  total other sections
81298  total positive sections
80675  total negative sections
4620  total tcpIDs containing positive predictions 


,tcpID,curr_div_idx,group
0,B15663,1,sermon
1,B15663,3,sermon
2,B15663,4,sermon
3,B15663,5,sermon
4,B15663,6,sermon
...,...,...,...
81293,A86287,8 | 22612,unknown
81294,A86287,8 | 22617,unknown
81295,A86287,10 | 22648,unknown
81296,A85699,4,unknown


In [174]:
4602/8205

0.5608775137111517

In [171]:
13785/15480, 67513/146493

(0.8905038759689923, 0.460861611135003)

In [139]:
# Feature importance
importances = classifier.feature_importances_  # shape (n_features,)
top_idx = np.argsort(importances)[::-1]
top_importances = importances[top_idx]
importance_df = []
for i, score in zip(top_idx, top_importances):
    importance_df.append({'index':i, 'feature': features[i], 'score': score})
importance_df = pd.DataFrame(importance_df)
importance_df.to_csv("../OUTPUTS/feature_importance.csv",index=False)
importance_df[:50]

,index,feature,score
0,10,TOP_np,0.057436
1,177,god (np1),0.047645
2,180,christ (np1),0.038741
3,753,christ,0.035720
4,217,god (npg1),0.026103
5,755,rom.,0.021772
6,272,cor. (np1),0.020766
7,271,rom. (np1),0.020458
8,248,scripture (n1),0.019273
9,352,2. (crd),0.018275


In [140]:
# cumulative importance — num features needed for percent total importance
sorted_imps = np.sort(importances)[::-1]
cumulative = np.cumsum(sorted_imps)
for pct in [0.5, 0.75, 0.9, 0.95, 0.99, 0.999]:
    n_needed = np.searchsorted(cumulative, pct) + 1
    print(f"{pct*100:.0f}% importance reached with {n_needed} features")

50% importance reached with 26 features
75% importance reached with 83 features
90% importance reached with 249 features
95% importance reached with 358 features
99% importance reached with 541 features
100% importance reached with 750 features
